# Linear Response Function for LW/SW and vertical velocity

### Imports

In [ ]:
import os

# Limit BLAS/OpenMP before importing NumPy. Restart the kernel after changing this.
for variable in (
    "OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS",
    "NUMEXPR_NUM_THREADS", "VECLIB_MAXIMUM_THREADS", "BLIS_NUM_THREADS",
):
    os.environ[variable] = "1"

import netCDF4 as nc
import numpy as np

#### Load CloudSat data

In [ ]:
# Load data
with nc.Dataset("/data92/b11209013/CloudSat/DATA/QR_gridded_15layer_-5_5.nc", "r") as cs_ds:

    cs_coords: dict[str, np.ndarray] = {
        key: cs_ds.variables[key][...]
        for key in cs_ds.dimensions.keys()
    }

    LW: np.ndarray = cs_ds.variables["QLW"][...]
    SW: np.ndarray = cs_ds.variables["QSW"][...]

# Load ERA5 DATA
with nc.Dataset("/data92/b11209013/ERA5_GRIB/Data/tropical_-10_10/T.nc", "r") as t_ds:

    era5_coords: dict[str, np.ndarray] = {
        key: t_ds.variables[key][...].astype(np.float32)
        for key in t_ds.dimensions.keys()
    }

    lat_lim = (era5_coords["lat"] >= -5.0) & (era5_coords["lat"] <= 5.0)

    T: np.ndarray = t_ds.variables["T"][..., lat_lim, :].astype(np.float32)

with nc.Dataset("/data92/b11209013/ERA5_GRIB/Data/tropical_-10_10/W.nc", "r") as w_ds:

    omega: np.ndarray = w_ds.variables["W"][..., lat_lim, :].astype(np.float32)


#### Transform from $\omega$ to w

In [ ]:
# Calculate density
ρ: np.ndarray = era5_coords["plev"][None, :, None, None] / 287.5 / T

# Transform pressure velocity to w
w = -omega / ρ / 9.81

del ρ, omega

#### Pre-processing

In [ ]:
# remove time and zonal mean
w_anom : np.ndarray = w  - np.nanmean(w , axis=(0, -1), keepdims=True)
LW_anom: np.ndarray = LW - np.nanmean(LW, axis=(0, -1), keepdims=True)
SW_anom: np.ndarray = SW - np.nanmean(SW, axis=(0, -1), keepdims=True)

# transform data to (nz, nx, nt, ny)
w_trans : np.ndarray = w_anom.transpose(1, 3, 0, 2)
LW_trans: np.ndarray = LW_anom.transpose(1, 3, 0, 2)
SW_trans: np.ndarray = SW_anom.transpose(1, 3, 0, 2)

del w_anom, LW_anom, SW_anom

# Keep only columns for which every layer is valid in w, LW, and SW.
# np.ma.filled handles NetCDF masked values; np.isfinite also rejects NaN/Inf.
w_finite : np.ndarray = np.isfinite(np.ma.filled(w_trans, np.nan))
LW_finite: np.ndarray = np.isfinite(np.ma.filled(LW_trans, np.nan))
SW_finite: np.ndarray = np.isfinite(np.ma.filled(SW_trans, np.nan))
valid_mask: np.ndarray = np.all(w_finite & LW_finite & SW_finite, axis=0)

# valid_mask has shape (nx, nt, ny). The regression cell collects only
# its capped train/test subsets instead of copying every valid column.

#### Calculate linear response function

In [ ]:
# Reserve the final time steps for a genuinely out-of-sample test.
test_fraction: float = 0.20
nt: int = valid_mask.shape[1]
split_time: int = min(max(int(np.floor((1.0 - test_fraction) * nt)), 1), nt - 1)
time_index: np.ndarray = np.arange(nt)[None, :, None]
train_mask: np.ndarray = valid_mask & (time_index < split_time)
test_mask : np.ndarray = valid_mask & (time_index >= split_time)

# Bound CPU and memory use. Sampling is reproducible, and test samples still
# come exclusively from the final, unseen 20% of time steps.
max_train_samples: int = 100_000
max_test_samples : int = 25_000

def limit_samples(mask: np.ndarray, maximum: int, seed: int) -> np.ndarray:
    positions = np.flatnonzero(mask)
    if positions.size <= maximum:
        return mask
    rng = np.random.default_rng(seed)
    selected = rng.choice(positions, size=maximum, replace=False)
    limited = np.zeros(mask.size, dtype=bool)
    limited[selected] = True
    return limited.reshape(mask.shape)

fit_mask : np.ndarray = limit_samples(train_mask, max_train_samples, seed=0)
eval_mask: np.ndarray = limit_samples(test_mask, max_test_samples, seed=1)

def collect_samples(mask: np.ndarray) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    # Return conventional regression layout: (n_sample, nz).
    return (
        np.asarray(w_trans[:, mask], dtype=np.float64).T,
        np.asarray(LW_trans[:, mask], dtype=np.float64).T,
        np.asarray(SW_trans[:, mask], dtype=np.float64).T,
    )

X_train, LW_train, SW_train = collect_samples(fit_mask)
X_test, LW_test, SW_test = collect_samples(eval_mask)

nz: int = X_train.shape[1]
if X_train.shape[0] < nz or X_test.shape[0] == 0:
    raise ValueError("The temporal split does not contain enough valid samples")

# Center and scale using training statistics only. This makes the ridge
# penalty comparable among vertical levels and prevents test-data leakage.
x_mean: np.ndarray = X_train.mean(axis=0)
x_scale: np.ndarray = X_train.std(axis=0)
if np.any(x_scale == 0.0):
    raise ValueError("At least one w level has zero variance in the training set")
X_scaled: np.ndarray = (X_train - x_mean) / x_scale

# One SVD is reused for both multi-output ridge regressions.
U, singular_values, Vt = np.linalg.svd(X_scaled, full_matrices=False)
condition_number: float = (
    singular_values[0] / singular_values[-1]
    if singular_values[-1] > np.finfo(float).eps else np.inf
)
alpha_grid: np.ndarray = singular_values[0]**2 * np.logspace(-8, 2, 51)

def fit_ridge_gcv(Y: np.ndarray) -> tuple[np.ndarray, np.ndarray, float]:
    """Fit all output levels and select alpha by training-only GCV."""
    y_mean: np.ndarray = Y.mean(axis=0)
    Y_centered: np.ndarray = Y - y_mean
    UtY: np.ndarray = U.T @ Y_centered
    projected_energy: np.ndarray = np.sum(UtY**2, axis=1)
    total_energy: float = float(np.sum(Y_centered**2))

    gcv_scores: list[float] = []
    for alpha in alpha_grid:
        shrinkage = singular_values**2 / (singular_values**2 + alpha)
        residual_ss = total_energy + np.sum(
            (shrinkage**2 - 2.0 * shrinkage) * projected_energy
        )
        effective_df = np.sum(shrinkage)
        gcv_scores.append(
            max(float(residual_ss), 0.0)
            / Y.size
            / (1.0 - effective_df / Y.shape[0])**2
        )

    alpha: float = float(alpha_grid[np.argmin(gcv_scores)])
    ridge_gain = singular_values / (singular_values**2 + alpha)
    coef_scaled: np.ndarray = (Vt.T * ridge_gain) @ UtY

    # Convert standardized coefficients back to physical coordinates:
    # Q[:, sample] = operator @ w[:, sample] + intercept.
    operator: np.ndarray = (coef_scaled / x_scale[:, None]).T
    intercept: np.ndarray = y_mean - operator @ x_mean
    return operator, intercept, alpha

L_LW, b_LW, alpha_LW = fit_ridge_gcv(LW_train)
L_SW, b_SW, alpha_SW = fit_ridge_gcv(SW_train)

LW_pred: np.ndarray = X_test @ L_LW.T + b_LW
SW_pred: np.ndarray = X_test @ L_SW.T + b_SW

def regression_metrics(actual: np.ndarray, predicted: np.ndarray) -> tuple[float, float]:
    residual_ss = np.sum((actual - predicted)**2)
    total_ss = np.sum((actual - actual.mean(axis=0))**2)
    rmse = float(np.sqrt(np.mean((actual - predicted)**2)))
    r2 = float(1.0 - residual_ss / total_ss) if total_ss > 0.0 else np.nan
    return rmse, r2

LW_rmse, LW_r2 = regression_metrics(LW_test, LW_pred)
SW_rmse, SW_r2 = regression_metrics(SW_test, SW_pred)

print(
    f"available train/test columns: {train_mask.sum():,}/{test_mask.sum():,}; "
    f"used: {X_train.shape[0]:,}/{X_test.shape[0]:,}"
)
print(f"standardized w condition number: {condition_number:.3e}")
print(f"LW: alpha={alpha_LW:.3e}, test RMSE={LW_rmse:.4g}, test R²={LW_r2:.4f}")
print(f"SW: alpha={alpha_SW:.3e}, test RMSE={SW_rmse:.4g}, test R²={SW_r2:.4f}")
print(f"operator shapes: L_LW={L_LW.shape}, L_SW={L_SW.shape}")